# Part 2 - Exploratory Data Analysis (EDA)

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sqlite3

## 1. Data Loading

In [2]:
conn = sqlite3.connect("../data/reviews_analysis.db")

hotels_df = pd.read_sql(
    "SELECT name FROM sqlite_master WHERE type='table'",
    conn
)

### Convert to pandas dataframe

In [3]:
tables = hotels_df['name'].tolist()
tables

['reviews', 'authors', 'hotels']

In [4]:
dataset = {}

for table in tables:
    query = f"SELECT * FROM {table}"
    dataset[table] = pd.read_sql(query, conn)

In [5]:
dataset.keys()

dict_keys(['reviews', 'authors', 'hotels'])

In [6]:
for table in dataset.keys():
    print(f'\nTable name:   {table}')
    print(f'{dataset[table].dtypes}')


Table name:   reviews
title                          str
text                           str
date_stayed                    str
hotel_id                     int64
num_helpful_votes            int64
review_date                    str
review_id                    int64
via_mobile                   int64
service_rating             float64
cleanliness_rating         float64
overall_rating             float64
value_rating               float64
location_rating            float64
sleep_quality_rating       float64
rooms_rating               float64
check_in_service_rating    float64
business_service_rating    float64
author_key                     str
dtype: object

Table name:   authors
username                 str
num_cities           float64
num_helpful_votes    float64
num_reviews          float64
num_type_reviews     float64
id                       str
location                 str
author_key               str
dtype: object

Table name:   hotels
hotel_id                         int64
num

### Converting stayed_at to datetime

In [7]:
dataset['reviews']

,title,text,date_stayed,hotel_id,num_helpful_votes,review_date,review_id,via_mobile,service_rating,cleanliness_rating,overall_rating,value_rating,location_rating,sleep_quality_rating,rooms_rating,check_in_service_rating,business_service_rating,author_key
0,“Big Disappointment!”,Since we have not traveled to San Francisco in...,October 2012,81416,2,2012-10-29,143991612,0,1.0,2.0,1.0,1.0,1.0,1.0,1.0,NaN,NaN,2ED506AD2366FA9781ECB11D0709C941|jhuredhead
1,“Very impressed!!”,We stayed on the 4th floor facing the road in ...,November 2011,2154898,0,2011-11-26,121053587,1,5.0,5.0,5.0,5.0,4.0,NaN,5.0,NaN,NaN,313701D811FA927A96FDD33430E168A2|BigJimDevine
2,“Near pefection”,"Fantastic. As a world traveler, I have stayed ...",April 2012,665258,0,2012-05-16,130065325,0,5.0,4.0,5.0,4.0,3.0,5.0,5.0,NaN,NaN,98209DB505D45C15A7EC50A2C6E528D0|Jasmin154
3,“Excellent Bar Staff”,The Polar Bar staff is the best in the country...,February 2012,677260,0,2012-02-21,125001925,0,5.0,5.0,5.0,5.0,5.0,5.0,5.0,NaN,NaN,6FB24D5E21E0DB99DE7C4BF8EBE1ED4F|John Z
4,“Naja”,Wir hatten für 3 Nächte 2 Zimmer gebucht. Die ...,February 2010,113923,0,2010-02-23,56889304,0,3.0,3.0,3.0,3.0,5.0,3.0,4.0,NaN,NaN,A8ECCAD48F0898C78DB3D6FCFE346D01|sigib80
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
79995,“Centrally located Ageing Hotel”,The hotel is in a downtown location and is wit...,October 2012,89602,0,2012-10-10,142516580,0,3.0,3.0,3.0,2.0,4.0,2.0,3.0,NaN,NaN,148A4837B65B265C6C4A259253E940FC|Andrew255103
79996,"“Very nice, but imperfect”",The room was very quiet - too quiet! Even the ...,September 2010,89585,0,2010-09-15,79670549,0,5.0,5.0,4.0,3.0,5.0,4.0,4.0,NaN,NaN,3567BCFCC2123EBFFBA85417C3A9A5D3|BusterCT1K
79997,“Great Value”,Totally recommend this place. Got a great nigh...,October 2012,226753,0,2012-10-24,143602018,0,5.0,5.0,5.0,5.0,5.0,5.0,5.0,NaN,NaN,0902B2912745CB0DF55C57B9B34D2D39|CYNTHIA M
79998,“Doing things right”,Just got back from a great NYC long weekend. T...,December 2012,1456411,0,2012-12-12,147288297,0,5.0,5.0,5.0,4.0,5.0,4.0,4.0,NaN,NaN,F5CEC1E7C9CE9CB9A000CCC9A0F184DE|cdbeach


In [8]:
dataset['reviews']['date_stayed'] = pd.to_datetime(
    dataset['reviews']['date_stayed'],
    format='%B %Y',
    errors='coerce'
)
dataset['reviews']['review_date'] = pd.to_datetime(
    dataset['reviews']['review_date'],
    format='%Y-%m-%d',
    errors='coerce'
)

In [16]:
dataset['reviews']['date_stayed'].isna().sum()
dataset['reviews'].dropna(subset='date_stayed', inplace=True)

In [18]:
print(f'length of data: {len(dataset['reviews'])}')

length of data: 79997


In [23]:
print(dataset['reviews']['review_date'].min())
print(dataset['reviews']['review_date'].max())

reviews_df = dataset['reviews']

filtered = reviews_df[reviews_df['review_date'] - reviews_df['date_stayed'] <= pd.Timedelta(days=150)]
print(f'length of data: {len(filtered)}')

2008-01-01 00:00:00
2012-12-20 00:00:00
length of data: 74117


In [ ]:
dataset['reviews'].head()

In [ ]:
type(dataset['reviews']['review_date'][0])

In [ ]:
dataset['reviews']['date_stayed'] = pd.to_datetime(dataset['reviews']['date_stayed'], format='%B %Y')

The dates data are not in the right type, so we must parse it.
These are some notable warnings that I noticed:
* date_stayed = should be in date (only the year and month that is important)
* review_date = should be in date (full format)


In [ ]:
# Converting date_stayed to a date format
dataset['reviews']['date_stayed_dt'] = pd.to_datetime(dataset['reviews']['date_stayed'], format='%B %Y')

# Converting review_date to a date format
dataset['reviews']['review_date'] = pd.to_datetime(dataset['reviews']['review_date'])

In [ ]:

print(f'\nTable name:   reviews')
print(f'{dataset['reviews'].dtypes}')

---

## 2. Data Exploration & Feature Engineering

### Reviews table

In [ ]:
reviews_df = dataset['reviews']
reviews_df.head()

In [ ]:
print(f'Reviews table has {len(reviews_df)} entries.')

reviews_df[reviews_df['date_stayed_dt'] == '2003-09-01']

The customer stayed on September 2003, but the review was given on 25 August 2008 which is about 5 years later. This data seems a little but irrelevant. There for we should check for the rest of the data, as there may be some wrong entries where the review_date is even earlier than the date_stayed.

In [ ]:
# Check for data where review_date is earlier than date_stayed_dt
invalid_entries = reviews_df[reviews_df['review_date'] < reviews_df['date_stayed_dt']].index
reviews_df.drop(invalid_entries,inplace=True)

print(f'Number of invalid data: {len(invalid_entries):,} entries')
print(f'Number of entries after removal: {len(reviews_df):,} entries')
# invalid_entries

Now that the data date is valid, we can add new feature called 'review_days_since_stay' that denotes how long the review is given after the stay date.

In [ ]:
reviews_df['reviews_days_since_stay'] = reviews_df['review_date'] - reviews_df['date_stayed_dt']
reviews_df.head(2)

In [ ]:
# Date
# reviews_df['review_date', 'date_stayed_dt']
print('======== DATE ANALYSIS ========')
print(f'date_stayed date range:    {reviews_df['date_stayed_dt'].min().strftime('%B %Y')} - {reviews_df['date_stayed_dt'].max().strftime('%B %Y')}')
print(f'review_date date range:    {reviews_df['review_date'].min().strftime('%-d %B %Y')} - {reviews_df['review_date'].max().strftime('%-d %B %Y')}')

In [ ]:
# Categorical
print('======== CATEGORICAL ANALYSIS ========')
print(f'- {reviews_df['via_mobile'].mean():.2%} of people fill the review via mobile')

for col in ['service_rating', 'cleanliness_rating', 'overall_rating', 'value_rating', 'location_rating', 'sleep_quality_rating', 'rooms_rating', 'check_in_service_rating', 'business_service_rating']:
    print(f'\n- {col} average of all hotel is {reviews_df[col].mean():.1f} stars')
    print(f'    - {(len(reviews_df[reviews_df[col] == 5]) / len(reviews_df)):.2%} of the rating is 5 stars')
    print(f'    - {(len(reviews_df[reviews_df[col] == 1]) / len(reviews_df)):.2%} of the rating is 1 stars')
    
    nan_count = reviews_df[col].isna().sum()
    if nan_count > 0:
        print(f'    - {(nan_count / len(reviews_df)):.2%} ({nan_count} entries) does not give any rating')

reviews_df[[
    'via_mobile',
    'service_rating',
    'cleanliness_rating',
    'overall_rating',
    'value_rating',
    'location_rating',
    'sleep_quality_rating',
    'rooms_rating',
    'check_in_service_rating',
    'business_service_rating'
    ]].describe()


In [ ]:
# Numerical
print('======== NUMERICAL ANALYSIS ========')
reviews_df[[
    'num_helpful_votes'
    ]].describe()

In [ ]:
rating_column = [
    'num_helpful_votes',
    'via_mobile',
]

fig, ax = plt.subplots(nrows=1, ncols=2, figsize=(12,6))
ax = ax.flatten()

for i, col in enumerate(rating_column):
    mean = reviews_df[col].mean()
    reviews_df_filled = reviews_df[col].fillna('NaN').astype(str)
    counts = reviews_df_filled.value_counts().sort_index(key=lambda x: pd.to_numeric(x, errors='coerce'))
    
    
    bars = ax[i].bar(counts.index, counts.values)
    
    # Add counts on top of each bar
    for bar in bars:
        height = bar.get_height()
        ax[i].text(bar.get_x() + bar.get_width()/2, height + 1, str(height), ha='center', va='bottom', fontsize=10)
    
    if col == 'num_helpful_votes':
        ax[i].axvline(mean, color='red', linestyle='--', label='Mean')
        ax[i].legend()
        
    ax[i].set_title(f'{col}')
    ax[i].set_xlabel('Num of Helpful vote')
    ax[i].set_ylabel('Count')

ax[0].grid(linestyle = '--', linewidth = 0.5)

plt.suptitle('Rating distribution for each column')
plt.tight_layout()
plt.show()

In [ ]:
rating_column = [
    'service_rating',
    'cleanliness_rating',
    'overall_rating',
    'value_rating',
    'location_rating',
    'sleep_quality_rating',
    'rooms_rating',
    'check_in_service_rating',
    'business_service_rating'
]

fig, ax = plt.subplots(nrows=3, ncols=3, figsize=(16,12))
ax = ax.flatten()

for i, col in enumerate(rating_column):
    mean = reviews_df[col].mean()
    
    reviews_df_filled = reviews_df[col].fillna('NaN').astype(str)
    counts = reviews_df_filled.value_counts()
    
    counts = counts.reindex(['1.0', '2.0', '3.0', '4.0', '5.0', 'NaN'], fill_value=0)
    colors = ['C0'] * 5 + ['gray']
    
    bars = ax[i].bar(counts.index, counts.values, color=colors)
    
    # Add counts on top of each bar
    for bar in bars:
        height = bar.get_height()
        ax[i].text(bar.get_x() + bar.get_width()/2, height + 1, str(height), ha='center', va='bottom', fontsize=10)
    
    # Set the alpha of the NaN to 0.7
    bars[-1].set_alpha(0.5)
    
    # Map numeric mean to bar position
    x_labels = ['1.0', '2.0', '3.0', '4.0', '5.0']
    x_positions = np.arange(len(x_labels))
    
    # Only draw mean if numeric, ignore NaN
    if not np.isnan(mean):
        # Find position on x-axis
        mean_pos = mean - 1  # Subtract position by 1
        ax[i].axvline(mean_pos, color='red', linestyle='--', label='Mean')
        ax[i].legend()
        
    ax[i].set_title(f'{col}')
    ax[i].set_xlabel('Rating')
    ax[i].set_ylabel('Count')
    ax[i].grid(linestyle = '--', linewidth = 0.5)

plt.suptitle('Rating distribution for each column', fontsize=18)
plt.tight_layout()
plt.show()

### Authors Table

In [ ]:
authors_df = dataset['authors']
authors_df.head()

In [ ]:
print(f'Number of entries in the authors dataframe: {len(authors_df):,} entries')

authors_df['num_reviews'].sum()
print('ALERT!!! Number of reviews in total are 101,958 but the reviews_sample.db that i am using only has 6,000. this data should not be available in real scenario, might cause data leakage')

In [ ]:
authors_df.describe()

In [ ]:
author_column = [
    'num_cities',
    'num_helpful_votes',
    'num_reviews',
    'num_type_reviews'
]

fig, ax = plt.subplots(nrows=2, ncols=2, figsize=(10,10))
ax = ax.flatten()

for i, col in enumerate(author_column):
    mean = authors_df[col].mean()
    
    authors_df_filled = authors_df[col].fillna('NaN').astype(str)
    counts = authors_df_filled.value_counts()
    
    
    counts, bin_edges, bars = ax[i].hist(
        authors_df[col],
        bins=8,
        edgecolor='black'
    )

    # Add count labels on top of bars
    for count, bar in zip(counts, bars):
        if count > 0:
            ax[i].text(
                bar.get_x() + bar.get_width()/2,
                count,
                int(count),
                ha='center',
                va='bottom',
                fontsize=9
            )

    # Mean line
    ax[i].axvline(mean, color='red', linestyle='--', label='Mean')
    ax[i].legend()

    ax[i].set_title(col)
    ax[i].set_xlabel(col)
    ax[i].set_ylabel('Count')
    ax[i].grid(linestyle='--', linewidth=0.5)

plt.suptitle('Author statistics distribution (binned)', fontsize=18)
plt.tight_layout()
plt.show()

### Hotels table

In [ ]:
hotels_df = dataset['hotels']
hotels_df.head()

In [ ]:
hotel_column = [
    'num_reviews',
]

fig, ax = plt.subplots(nrows=1, ncols=1, figsize=(5,5))

for i, col in enumerate(hotel_column):
    print(col)
    mean = hotels_df[col].mean()
    
    hotels_df_filled = hotels_df[col].fillna('NaN').astype(str)
    counts = hotels_df_filled.value_counts()
    
    
    counts, bin_edges, bars = ax.hist(
        hotels_df[col],
        bins=8,
        edgecolor='black'
    )

    # Add count labels on top of bars
    for count, bar in zip(counts, bars):
        if count > 0:
            ax.text(
                bar.get_x() + bar.get_width()/2,
                count,
                int(count),
                ha='center',
                va='bottom',
                fontsize=9
            )

    # Mean line
    ax.axvline(mean, color='red', linestyle='--', label='Mean')
    ax.legend()

    ax.set_title(col)
    ax.set_xlabel(col)
    ax.set_ylabel('Count')
    ax.grid(linestyle='--', linewidth=0.5)

plt.suptitle('Hotels statistics distribution (binned)')
plt.tight_layout()
plt.show()

In [ ]:
hotel_column = [
    'avg_service_rating',
    'avg_cleanliness_rating',
    'avg_overall_rating',
    'avg_value_rating',
    'avg_location_rating',
    'avg_sleep_quality_rating',
    'avg_rooms_rating',
]

fig, ax = plt.subplots(nrows=2, ncols=4, figsize=(20,10))
ax = ax.flatten()

for i, col in enumerate(hotel_column):
    mean = hotels_df[col].mean()
    
    hotels_df_filled = hotels_df[col].fillna('NaN').astype(str)
    counts = hotels_df_filled.value_counts()
    
    counts = counts.reindex(['1.0', '2.0', '3.0', '4.0', '5.0', 'NaN'], fill_value=0)
    colors = ['C0'] * 5 + ['gray']
    
    bars = ax[i].bar(counts.index, counts.values, color=colors)
    
    # Add counts on top of each bar
    for bar in bars:
        height = bar.get_height()
        ax[i].text(bar.get_x() + bar.get_width()/2, height + 1, str(height), ha='center', va='bottom', fontsize=10)
    
    # Set the alpha of the NaN to 0.7
    bars[-1].set_alpha(0.5)
    
    # Map numeric mean to bar position
    # x_labels = ['1.0', '2.0', '3.0', '4.0', '5.0']
    # x_positions = np.arange(len(x_labels))
    
    # Only draw mean if numeric, ignore NaN
    if not np.isnan(mean):
        # Find position on x-axis
        mean_pos = mean - 1  # Subtract position by 1
        ax[i].axvline(mean_pos, color='red', linestyle='--', label='Mean')
        ax[i].legend()
        
    ax[i].set_title(f'{col}')
    ax[i].set_xlabel('Rating')
    ax[i].set_ylabel('Count')
    ax[i].grid(linestyle = '--', linewidth = 0.5)

plt.suptitle('Rating distribution for each column', fontsize=18)
plt.tight_layout()
plt.show()

---

## 3. Feature Engineering

**is_frequent_reviewer** in the authors_df

**helpful_votes_per_review** in the authors_df

**avg_stars_given** in the authors_df

**ratings_variance** in the hotels_df to make sure the rating is consistent

**distance_from_avg** in the reviews_df ratings

---

## 4. Analytics by Business Questions

### **a. How does the hotel rating differ by location?**

ALERT!!! Location is not in the hotels_df

### **b. Best and worst hotels from each service category?**

### **c. Repeating visitor? mobile vs non mobile? fast and late review (3 mos threshold)?**

### **d. Best compliment Keyword for each hotels?**

My idea is to normalize the reviews, and check using tf-idf

In [ ]:
#

### **e. Which rating category contributes the most to the overall rating?**

check the correlation of the scores. this can help the hotel to decide where to invest next

### **f. Trend in the stars during the years for each hotel or for each location group**

### **g. "How does my hotel perform?"**

by inputting the average rating of the current hotel, we can show graph like "you perform better than 90% of hotels worldwide, and 95% in the same region".

### **h. top 5 hotel and bottom 5 hotel per location**

### **i. best and worst service per location to tell which countries are better in certain category**

### **j. does mobile reviews is shorter and less helpful?**

### **k. Do fast reviews tend to be more negative?**

### **l. are experienced travelers more critical than casual reviewers?**

---

Column Explanation:
* title
* text
* date_stayed
* hotel_id
* num_helpful_votes
* review_date
* review_id
* via_mobile
* service_rating
* cleanliness_rating
* overall_rating
* value_rating
* location_rating
* sleep_quality_rating
* rooms_rating
* check_in_service_rating
* business_service_rating
* author_key

### Close connection

In [ ]:
conn.close()

Issue:
- is the location denotes the hotel's location or the user
- where is the complete dataset .db
- data validation step is still incomplete (date checking and else)
- column description step is missing yet essential for us to make
- feature engineering part needs more ideas